# Lentils × Dinomaly — inference + per-class AUROC

Load a pipeline trained by one of the `lentils_*_train_tutorial.ipynb` notebooks, run it over
the **180-frame test split**, and report:

- **overall pixel + image AUROC** (binary anomaly = any foreign object), and
- a **per-class pixel AUROC** breakdown across the 8 COCO categories, using the baked
  `class_mask` (one-vs-background per class).

> **Prerequisites**
> - Train + save a pipeline first (run `lentils_rgb_train_tutorial.ipynb`, or CIR / adaclip-bands).
>   This notebook loads it from `PIPELINE_DIR` (a plain variable in the setup cell; defaults to the
>   RGB run's `outputs/lentils_rgb_run/trained_models`). Edit it to evaluate a different run.
> - **Data** — `prepare_lentils_data` downloads + converts the dataset to per-frame NPZ under
>   `outputs/npz_local` (reused if the RGB train notebook already produced it there), writing a
>   `universe.csv` + `splits.json`; the 180-frame test split is resolved from the splits.json.
> - `TEST_LIMIT` (0 = full 180; N = first N frames) and `SMOKE_LIMIT` (limit frames per split when
>   converting) are plain variables in the setup cell.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

sys.path.insert(0, str(Path(".").resolve()))
import utils
from utils import LENTILS_CATEGORIES, prepare_lentils_data, resolve_config, resolve_pipeline

config = resolve_config()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Which trained run to evaluate + how much (edit these) ---------------
PIPELINE_DIR = Path("outputs/lentils_rgb_run/trained_models")  # a train notebook's output dir
TEST_LIMIT = 0  # 0 = full 180-frame test split; N = first N frames
SMOKE_LIMIT = 0  # 0 = all frames; N = N per split when converting

PIPE_YAML, PIPE_PT = resolve_pipeline(PIPELINE_DIR)
print("Device:        ", DEVICE)
print("Pipeline YAML: ", PIPE_YAML)
print("Weights:       ", PIPE_PT, f"({PIPE_PT.stat().st_size / 1e6:.0f} MB)")

## 1 · Load the trained pipeline

Register the Dinomaly plugin (so the saved node classes resolve), then
`CuvisPipeline.load_pipeline` rebuilds the graph and loads the weights.

In [ ]:
from cuvis_ai_core.pipeline.pipeline import CuvisPipeline
from cuvis_ai_core.utils.node_registry import NodeRegistry
from cuvis_ai_dataloader.data import MultiNpzDataModule
from cuvis_ai_schemas.training import DataSplitConfig

registry = NodeRegistry()
registry.register_plugin(str(config["plugins_yaml"]))
pipeline = CuvisPipeline.load_pipeline(
    str(PIPE_YAML), weights_path=str(PIPE_PT), device=DEVICE, node_registry=registry
)
pipeline.torch_layers.eval()
print("Loaded:", pipeline.name, "| nodes:", [n.name for n in pipeline.nodes])

SPLITS_JSON, UNIVERSE_CSV = prepare_lentils_data(Path("outputs/npz_local"), limit=SMOKE_LIMIT)
datamodule = MultiNpzDataModule(
    splits=DataSplitConfig(splits_path=str(SPLITS_JSON.resolve())),
    universe_csv=str(UNIVERSE_CSV),
    batch_size=1,
    num_workers=0,
)
datamodule.setup(stage="test")
print("Test frames:", len(datamodule.test_ds))

### Pipeline graph

Inline view of the restored graph. A `CuvisPipeline` renders itself in Jupyter as an
inline SVG, rendered from Graphviz DOT in memory (needs the system `dot` binary; it falls
back to a Mermaid source block otherwise).

In [ ]:
pipeline

## 2 · Run inference on the test split

`utils.run_test_inference` runs the pipeline forward per frame (DINOv2 encoder under
`torch.no_grad()`) and collects the anomaly score map, image score, and baked `mask` /
`class_mask` — reusing the same extraction helpers as `examples/run_saved_dinomaly_pipeline_test_npz.py`.

In [ ]:
results = utils.run_test_inference(
    pipeline, datamodule, device=torch.device(DEVICE), limit=TEST_LIMIT
)
n_anom = sum(1 for r in results if r["mask"] is not None and r["mask"].any())
print(f"{len(results)} frames evaluated | {n_anom} anomalous, {len(results) - n_anom} normal")

## 3 · Overall metrics

Pixel AUROC pools every pixel (positive = foreign object); image AUROC uses the per-frame
anomaly score (positive = frame contains any foreign object).

In [ ]:
overall = utils.overall_auroc(results)
for k, v in overall.items():
    print(f"{k:14s}: {v:.4f}")
if not overall:
    print(
        "(need both normal and anomalous frames for AUROC — increase TEST_LIMIT / drop SMOKE_LIMIT)"
    )

## 4 · Per-class pixel AUROC

For each non-background category, a one-vs-background pixel AUROC (this class's pixels as
positives, normal pixels as negatives), read straight from the baked `class_mask`. Classes
absent from the evaluated frames are skipped.

In [ ]:
scores = [r["score_map"] for r in results if r["score_map"] is not None]
cmasks = [r["class_mask"] for r in results if r["class_mask"] is not None]
per_class = utils.per_class_pixel_auroc(scores, cmasks, LENTILS_CATEGORIES)
for name, v in sorted(per_class.items(), key=lambda kv: kv[1]):
    print(f"{name:12s} {v:.4f}")
if per_class:
    utils.plot_per_class_auroc_bar(per_class, title="Lentils per-class pixel AUROC")
    plt.show()
else:
    print("(no non-background classes present in the evaluated frames)")

## 5 · Qualitative panels

A few anomalous frames: false-color scene, anomaly heatmap, and the scene with the
ground-truth contour overlaid. Cubes are reloaded from their NPZ for a true false-color view.

In [ ]:
anom = [r for r in results if r["mask"] is not None and r["mask"].any()][:3]
for r in anom:
    if r.get("path"):
        fr = utils.load_lentils_frame(r["path"])
        cube, wl = fr["cube"], fr["wavelengths"]
    else:  # fall back to the selector's RGB output
        cube = r["rgb"][None] if r["rgb"] is not None else None
        wl = np.array([650, 550, 450])
    title = f"index={r.get('index')}  score={r.get('anomaly_score')}"
    if cube is not None:
        utils.render_inference_panel(
            cube if cube.ndim == 3 else cube[0],
            r["score_map"],
            wavelengths=wl,
            gt_mask=r["mask"],
            title=title,
        )
        plt.show()

## Takeaways

- The **local** path is verified end-to-end. Numbers come from *your* trained pipeline — a
  1-epoch smoke will look weak; run the full 50 epochs (train notebook with
  `LENTILS_MAX_EPOCHS=50`, or `examples/train_dinomaly_rgb_multifile.py`) for a real result.
- Per-class AUROC surfaces which foreign-object types Dinomaly separates best from normal
  lentils — useful for deciding where a supervised head would add the most.
- Swap `LENTILS_PIPELINE_DIR` to compare RGB vs CIR vs AdaCLIP-bands runs on the identical test set.